<a href="https://colab.research.google.com/github/sabbir179/Multi_Agent_Medical_AI/blob/master/notebooks/dataset_verification_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A Multi-Agent Framework for Medical AI: Leveraging Fine-Tuned GPT, LLaMA, and DeepSeek R1 for Evidence-Based and Bias-Aware Clinical Query Processin — Reproduction & Revision Work
**Original Framework & Research:** Naomi Nourmohammadi, Md Meem Hossain,
The Anh Han, Safina Showkat Ara, Zia Ush Shamszaman


**Supervisor:** Dr. Zia Ush Shamszaman
**Collaborator:** Md Meem Hossain

**This Notebook — Reproduction, Verification & Documentation:** Sabbir Ahmed

**Purpose:** Reproducing the framework for IP&M resubmission (IPM-D-25-04242R1)
following reviewer feedback. Original fine-tuned checkpoints are no longer
accessible (confirmed by original author) — this notebook documents full
reproduction from source code, dataset, and published settings.

## 1. Environment Setup

In [ ]:
!git clone https://github.com/sabbir179/Multi_Agent_Medical_AI
%cd Multi_Agent_Medical_AI

fatal: destination path 'Multi_Agent_Medical_AI' already exists and is not an empty directory.
/content/Multi_Agent_Medical_AI


In [ ]:
!pip install -r requirements.txt


ERROR: Could not find a version that satisfies the requirement gc (from versions: none)
ERROR: No matching distribution found for gc


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


True
Tesla T4


## 2. Reviewing Project Documentation
Checking the original README for setup instructions and model links.

In [ ]:
!cat README.md


# 🏥 Multi Agent Medical AI 🤖  
🚀 This is an AI-driven **medical question-answering system** integrating **LLaMA 3.1, DeepSeek R1, and Transformer models** to provide **accurate, evidence-based medical responses**.  
It is powered by **3 AI Agents**, **Evidence Retrieval AI Agent**, **Clinical Reasoning AI Agent**, **Uncertainty & Refinement AI Agent**, and features **Human-in-the-Loop (HITL) Review** for ensuring reliability and reducing AI bias.


---
## 🔥 What's Special About MedAI?
✅ **Fine-Tuned LLaMA & DeepSeek on 20,000+ Medical Data Points** 📊  
   - Trained on a **large-scale medical dataset** for **domain-specific accuracy**.  
   - Ensures **high-quality, expert-verified medical answers**.  

📌 **GitHub Link to Fine-Tuned Models:**  
🔗 **[Fine-Tuned LLaMA 3.1 Model](https://github.com/Naominour/Fine_Tuning_LLaMA_Model)**  
🔗 **[Fine-Tuned DeepSeek R1 Model](https://github.com/Naominour/Fine-tuning-DeepSeek-R1)**  

---

## 🤖 AI Agent Architecture (Human-in-the-Loop)
MedAI's r

## 3. Locating the Fine-Tuned Model Checkpoints
Cloning the two training repositories referenced in the README to check
whether trained model weights are available.

In [ ]:
%cd /content
!git clone https://github.com/Naominour/Fine_Tuning_LLaMA_Model.git
!git clone https://github.com/Naominour/Fine-tuning-DeepSeek-R1.git

/content
fatal: destination path 'Fine_Tuning_LLaMA_Model' already exists and is not an empty directory.
fatal: destination path 'Fine-tuning-DeepSeek-R1' already exists and is not an empty directory.


In [ ]:
!ls -lhR /content/Fine_Tuning_LLaMA_Model
!ls -lhR /content/Fine-tuning-DeepSeek-R1

/content/Fine_Tuning_LLaMA_Model:
total 212K
-rw-r--r-- 1 root root 4.2K Aug  5 00:51 app.py
-rw-r--r-- 1 root root  34K Aug  5 00:51 llama31.py
drwxr-xr-x 2 root root 4.0K Aug  5 00:51 llama-models
-rw-r--r-- 1 root root 4.3K Aug  5 00:51 medical_data.py
-rw-r--r-- 1 root root 6.0K Aug  5 00:51 README.md
-rw-r--r-- 1 root root  11K Aug  5 00:51 reference.py
-rw-r--r-- 1 root root 124K Aug  5 00:51 result.ipynb
drwxr-xr-x 2 root root 4.0K Aug  5 00:51 src
-rw-r--r-- 1 root root 5.6K Aug  5 00:51 tokenizer.py

/content/Fine_Tuning_LLaMA_Model/llama-models:
total 0

/content/Fine_Tuning_LLaMA_Model/src:
total 24K
-rw-r--r-- 1 root root   1 Aug  5 00:51 image
-rw-r--r-- 1 root root 20K Aug  5 00:51 result.png
/content/Fine-tuning-DeepSeek-R1:
total 272K
-rw-r--r-- 1 root root 3.6K Aug  5 00:51 DeepSeek-R1.py
-rw-r--r-- 1 root root  99K Aug  5 00:51 plots.png
-rw-r--r-- 1 root root 2.3K Aug  5 00:51 README.md
-rw-r--r-- 1 root root 161K Aug  5 00:51 response.png


## 4. Reviewing the LLaMA Training Script
Checking required data format, hyperparameters, and dependencies.

In [ ]:
!cat /content/Fine_Tuning_LLaMA_Model/llama31.py


import os
import glob
import fire
import time
import json
import math
import torch
import mlflow
import numpy as np
from torch import nn
import matplotlib.pyplot as plt
import torch.nn.functional as F

from pathlib import Path
from dataclasses import dataclass
from typing import List, Optional, Tuple, TypedDict
from torch import nn
from tokenizer import Tokenizer
# -----------------------------------------------------------------------------
# ModelArgs


@dataclass
class ModelArgs:
    """
    A dataclass for storing and configuring model hyperparameters.
    """
    dim: int = 4096  # Dimensionality of model embeddings
    n_layers: int = 32  # Number of transformer layers
    n_heads: int = 32  # Number of attention heads
    n_kv_heads: Optional[int] = None  # Number of key-value heads (default: equal to n_heads)
    vocab_size: int = -1  # Size of the vocabulary
    multiple_of: int = 256  # SwiGLU hidden layer size must be a multiple of this value
    ffn_dim_multiplier: Optional

## 5. Reviewing the DeepSeek Training Script
Checking required data format, hyperparameters, and dependencies.

In [ ]:
!cat /content/Fine-tuning-DeepSeek-R1/DeepSeek-R1.py

from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported # Check if the hardware support bfloat16 precision
from trl import SFTTrainer # Supervised Fine-Tuning Trainer
from huggingface_hub import login
from transformers import TrainingArguments # Define training hyperparameter
from datasets import load_dataset

import os
import torch
import wandb




# Initialise Hugging Face and Wandb tokens
huggingface_token = os.environ.get("HF_TOKEN")
wandb_token = os.environ.get("wab")

# Login to Hugginface & Wandb
login(huggingface_token)
wandb.login(key=wandb_token)
run = wandb.init(
    project="Fine-tuning DeepSeek-R1-Distill-Llama-8B on Medical Dataset",
    job_type="training",
    anonymous="allow"
)

# Set parameters
max_seq_length = 2048 # Max sequence length the model can handle
dtype = None
load_in_4bit = True # 4-bit Quantisation

# Load model & tokenizer using Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Di

## 6. Dataset Acquisition
Downloading MedQuAD from the Kaggle source confirmed by the original author.

In [ ]:
!pip install kagglehub
import kagglehub
path = kagglehub.dataset_download("pythonafroz/medquad-medical-question-answer-for-ai-research")
print(path)

Using Colab cache for faster access to the 'medquad-medical-question-answer-for-ai-research' dataset.
/kaggle/input/medquad-medical-question-answer-for-ai-research


In [ ]:
!ls -lh {path}


total 22M
-rw-r--r-- 1 1000 1000 22M Aug  5 00:51 medquad.csv


## 7. Dataset Inspection
Checking structure, column names, and row count.

In [ ]:
import pandas as pd
df = pd.read_csv(f"{path}/medquad.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

(16412, 4)
['question', 'answer', 'source', 'focus_area']


,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


## 8. Data Preprocessing
Renaming columns to match the training script format, removing duplicates,
and creating an 80/20 train/test split (seed=3407) for reproducibility.

In [ ]:
from sklearn.model_selection import train_test_split

# Rename columns to match training script
df = df.rename(columns={"question": "Question", "answer": "Answer"})

# Remove duplicates
df = df.drop_duplicates(subset=["Question", "Answer"]).reset_index(drop=True)
print("After dedup:", df.shape)

# Create train/test split (80/20, fixed seed for reproducibility)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=3407)
df.loc[train_df.index, "split"] = "train"
df.loc[test_df.index, "split"] = "test"

print(df["split"].value_counts())

# Save prepared file
df.to_csv("/content/medquad_prepared.csv", index=False)
print("Saved to /content/medquad_prepared.csv")

After dedup: (16364, 4)
split
train    13091
test      3273
Name: count, dtype: int64
Saved to /content/medquad_prepared.csv


## 9. Backing Up Prepared Data to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p /content/drive/MyDrive/MedicalAI_Revision
!cp /content/medquad_prepared.csv /content/drive/MyDrive/MedicalAI_Revision/
print("Backed up to Drive")

Backed up to Drive


## 10. Dataset Discrepancy Investigation
The paper reports over 20,000 QA pairs; the working dataset has 16,364
after deduplication. This section verifies the dataset against two
independent sources.

### 10.1 Counting the Full Original MedQuAD Dataset (XML source)

In [ ]:
%cd /content
!git clone https://github.com/abachaa/MedQuAD.git
!find /content/MedQuAD -name "*.xml" | wc -l

/content
Cloning into 'MedQuAD'...
remote: Enumerating objects: 11310, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 11310 (delta 7), reused 4 (delta 4), pack-reused 11300 (from 1)
Receiving objects: 100% (11310/11310), 11.01 MiB | 18.33 MiB/s, done.
Resolving deltas: 100% (6807/6807), done.
11274


### 10.2 Verifying Against an Independent CSV Conversion

In [ ]:
!git clone https://github.com/avery-lockwood/MedQuAD-CSVs.git
!find /content/MedQuAD-CSVs -name "*.csv" | wc -l
!ls -lh /content/MedQuAD-CSVs

Cloning into 'MedQuAD-CSVs'...
remote: Enumerating objects: 11347, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 11347 (delta 8), reused 5 (delta 5), pack-reused 11338 (from 1)
Receiving objects: 100% (11347/11347), 15.98 MiB | 17.92 MiB/s, done.
Resolving deltas: 100% (6822/6822), done.
12
total 984K
-rw-r--r--  1 root root  663 Aug  5 00:56 changes.md
drwxr-xr-x  2 root root 4.0K Aug  5 00:56 _data_as_csvs
drwxr-xr-x 14 root root 4.0K Aug  5 00:56 _data_as_xmls
-rw-r--r--  1 root root  17K Aug  5 00:56 LICENSE.txt
-rw-r--r--  1 root root 938K Aug  5 00:56 QA-TestSet-LiveQA-Med-Qrels-2479-Answers.zip
-rw-r--r--  1 root root 3.0K Aug  5 00:56 readme.txt
-rw-r--r--  1 root root 7.3K Aug  5 00:56 xml_to_csvs.ipynb


### 10.3 Counting QA Pairs Directly from Original XML

In [ ]:
import xml.etree.ElementTree as ET
import glob

xml_files = glob.glob("/content/MedQuAD/**/*.xml", recursive=True)
total_pairs = 0
for f in xml_files:
    try:
        tree = ET.parse(f)
        pairs = tree.findall(".//QAPair")
        total_pairs += len(pairs)
    except Exception as e:
        pass

print("Total XML files:", len(xml_files))
print("Total QA pairs:", total_pairs)

Total XML files: 11274
Total QA pairs: 47441


### 10.4 Summing Rows Across the 12 Source CSVs

In [ ]:
import pandas as pd
import glob

csv_files = glob.glob("/content/MedQuAD-CSVs/_data_as_csvs/*.csv")
print("CSV files found:", len(csv_files))

total_rows = 0
for f in csv_files:
    df = pd.read_csv(f)
    print(f.split("/")[-1], ":", len(df), "rows")
    total_rows += len(df)

print("\nTOTAL ROWS:", total_rows)

CSV files found: 12
12_MPlusHerbsSupplements_QA.csv : 792 rows
11_MPlusDrugs_QA.csv : 12889 rows
8_NHLBI_QA_XML.csv : 559 rows
9_CDC_QA.csv : 270 rows
4_MPlus_Health_Topics_QA.csv : 981 rows
2_GARD_QA.csv : 5394 rows
3_GHR_QA.csv : 5430 rows
1_CancerGov_QA.csv : 729 rows
6_NINDS_QA.csv : 1104 rows
5_NIDDK_QA.csv : 1192 rows
7_SeniorHealth_QA.csv : 769 rows
10_MPlus_ADAM_QA.csv : 17348 rows

TOTAL ROWS: 47457


### 10.5 Checking for Empty/Missing Answers (CSV source)
Testing whether the size difference is explained by entries with missing
answers, likely due to copyright restrictions on certain NIH sources.

In [ ]:
import pandas as pd
import glob

csv_files = glob.glob("/content/MedQuAD-CSVs/_data_as_csvs/*.csv")

empty_answers = 0
valid_answers = 0
for f in csv_files:
    df = pd.read_csv(f)
    empty_answers += df['Answer'].isna().sum() + (df['Answer'].astype(str).str.strip() == "").sum()
    valid_answers += df['Answer'].notna().sum()

print("Rows with empty/missing answers:", empty_answers)
print("Rows with actual answers:", valid_answers)

KeyError: 'Answer'

*Note: initial attempt referenced the column with capitalized/original
naming; corrected below after checking the actual column names in
this data source.*

In [ ]:
import pandas as pd
import glob

csv_files = glob.glob("/content/MedQuAD-CSVs/_data_as_csvs/*.csv")

# Check one file's columns first
sample = pd.read_csv(csv_files[0])
print("Columns in this file:", sample.columns.tolist())
print(sample.head(2))

Columns in this file: ['Unnamed: 0', 'question', 'question_id', 'question_type', 'answer']
   Unnamed: 0                               question question_id  \
0           0           What is Activated Charcoal ?   0000001-1   
1           1  How effective is Activated Charcoal ?   0000001-2   

         question_type  answer  
0          information     NaN  
1  how effective is it     NaN  


In [ ]:
import pandas as pd
import glob

csv_files = glob.glob("/content/MedQuAD-CSVs/_data_as_csvs/*.csv")

empty_answers = 0
valid_answers = 0
for f in csv_files:
    df = pd.read_csv(f)
    empty_answers += df['answer'].isna().sum() + (df['answer'].astype(str).str.strip() == "").sum()
    valid_answers += df['answer'].notna().sum() - (df['answer'].astype(str).str.strip() == "").sum()

print("Rows with empty/missing answers:", empty_answers)
print("Rows with actual (non-empty) answers:", valid_answers)
print("Total:", empty_answers + valid_answers)

Rows with empty/missing answers: 31034
Rows with actual (non-empty) answers: 16423
Total: 47457


### 10.6 Verifying Question-Level Content Overlap
Confirming whether every question in the working dataset also exists in
the original MedQuAD source.

In [ ]:
import xml.etree.ElementTree as ET
import glob

# Extract actual question texts from original XML
xml_files = glob.glob("/content/MedQuAD/**/*.xml", recursive=True)
xml_questions = set()
for f in xml_files:
    try:
        tree = ET.parse(f)
        for qpair in tree.findall(".//QAPair"):
            q = qpair.find(".//Question")
            if q is not None and q.text:
                xml_questions.add(q.text.strip().lower())
    except Exception:
        pass

print("Unique questions in original XML:", len(xml_questions))

# Compare against Kaggle questions
kaggle_questions = set(df["Question"].str.strip().str.lower())
print("Unique questions in Kaggle file:", len(kaggle_questions))

overlap = kaggle_questions & xml_questions
print("Questions matching between Kaggle and original XML:", len(overlap))
print("Questions in Kaggle NOT found in original XML:", len(kaggle_questions - xml_questions))

Unique questions in original XML: 43449


KeyError: 'Question'

*Note: initial attempt referenced the column with capitalized/original
naming; corrected below after checking the actual column names in
this data source.*

In [ ]:
# Reload the prepared Kaggle file to be safe
df_kaggle = pd.read_csv("/content/medquad_prepared.csv")
print("Kaggle columns:", df_kaggle.columns.tolist())

kaggle_questions = set(df_kaggle["Question"].str.strip().str.lower())
print("Unique questions in Kaggle file:", len(kaggle_questions))

overlap = kaggle_questions & xml_questions
print("Questions matching between Kaggle and original XML:", len(overlap))
print("Questions in Kaggle NOT found in original XML:", len(kaggle_questions - xml_questions))

Kaggle columns: ['Question', 'Answer', 'source', 'focus_area', 'split']
Unique questions in Kaggle file: 14349
Questions matching between Kaggle and original XML: 14349
Questions in Kaggle NOT found in original XML: 0


### 10.7 Independent Verification: Empty Answers Counted Directly from XML
Cross-checking Section 10.5's result using the original XML source directly,
rather than the third-party CSV conversion.

In [ ]:
empty_in_xml = 0
valid_in_xml = 0
for f in xml_files:
    try:
        tree = ET.parse(f)
        for qpair in tree.findall(".//QAPair"):
            a = qpair.find(".//Answer")
            if a is None or not a.text or not a.text.strip():
                empty_in_xml += 1
            else:
                valid_in_xml += 1
    except Exception:
        pass

print("Empty answers (from original XML directly):", empty_in_xml)
print("Valid answers (from original XML directly):", valid_in_xml)

Empty answers (from original XML directly): 31034
Valid answers (from original XML directly): 16407


## 10.8 Findings Summary

**Verification method:** Content and count comparisons were performed
against two independent sources — the original MedQuAD XML repository
(github.com/abachaa/MedQuAD) and a separate CSV conversion of the same
data (github.com/avery-lockwood/MedQuAD-CSVs) — not counts alone.

**Q1 — Pairs compared, not just counts (see Section 10.6):** Verified
that all 14,349 unique questions in the working (Kaggle) dataset match
questions found in the original MedQuAD XML source exactly. Zero
mismatches found.

**Q2 — Empty answers verified from original XML directly (see Sections
10.5 and 10.7):** Cross-checked using two methods — the CSV conversion
(Section 10.5: 31,034 empty / 16,423 valid) and direct XML parsing
(Section 10.7: 31,034 empty / 16,407 valid). Both align closely,
confirming the result independent of the CSV conversion tool.

**Q3 — Duplicate criteria (see Section 8):** Rows were considered
duplicates only when both the Question text AND Answer text were
identical (subset=["Question", "Answer"]). Not duplicate questions
alone.

**Q4 — Kaggle dataset origin (see Section 10.6):** Confirmed the working
dataset is a filtered subset of the original MedQuAD (all questions
present in the source XML), not a separately preprocessed or merged
dataset.

**Q5 — Notebook/script:** This notebook itself serves as the full
methodology record, shared via GitHub (https://github.com/sabbir179/Multi_Agent_Medical_AI/blob/master/notebooks/dataset_verification_notebook.ipynb$0).

**Conclusion:** The working dataset (16,364 pairs after deduplication)
represents the complete set of publicly available MedQuAD pairs with
valid answers. The paper's stated figure of "20,000+" pairs could not
be matched to any verifiable version of MedQuAD — full (47,457),
CSV-converted, or answered-only subsets.